<a href="https://colab.research.google.com/github/geeheen61-gif/Ai_crop_model/blob/main/Crop_ai_fine_tunning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!ls

sample_data  trained_model.h5  trained_model.keras  training_hist.json


In [1]:
!wget -O /content/plant_village.zip "https://data.mendeley.com/public-files/datasets/tywbtsjrjv/files/9e7a2e02-41d7-4a8c-80d8-9c7a102b4d22/file_downloaded"

--2025-11-26 10:40:08--  https://data.mendeley.com/public-files/datasets/tywbtsjrjv/files/9e7a2e02-41d7-4a8c-80d8-9c7a102b4d22/file_downloaded
Resolving data.mendeley.com (data.mendeley.com)... 162.159.133.86, 162.159.130.86
Connecting to data.mendeley.com (data.mendeley.com)|162.159.133.86|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1137 (1.1K) [application/json]
Saving to: ‘/content/plant_village.zip’

/content/plant_vill 100%[===================>]   1.11K  --.-KB/s    in 0s      

2025-11-26 10:40:09 (12.7 MB/s) - ‘/content/plant_village.zip’ saved [1137/1137]



In [2]:
import tensorflow_datasets as tfds

ds = tfds.load("plant_village", split="train", as_supervised=True)
# then process ds for your training


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/plant_village/incomplete.27YSCH_1.0.2/plant_village-train.tfrecord*...:   …

Dataset plant_village downloaded and prepared to /root/tensorflow_datasets/plant_village/1.0.2. Subsequent calls will reuse this data.


In [3]:
%%writefile /content/train_script.py
import os
import argparse
import tensorflow as tf
import json

# --- Paste your entire training script here ---
# (everything from build_dataset(), build_model(), fine_tune_model(), main(), etc.)


Writing /content/train_script.py


In [4]:
!python /content/train_script.py \
    --model_type disease \
    --data_dir /content/dataset \
    --img_size 224 \
    --epochs 15 \
    --fine_tune_epochs 8 \
    --batch_size 32 \
    --out_dir /content/output_models


2025-11-26 10:45:47.682485: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764153947.723964    1885 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764153947.736847    1885 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764153947.766346    1885 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764153947.766407    1885 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764153947.766420    1885 computation_placer.cc:177] computation placer alr

In [5]:
!mkdir -p /content/output_models


In [6]:
!python /content/train_script.py \
    --model_type disease \
    --data_dir /content/dataset \
    --img_size 224 \
    --epochs 15 \
    --fine_tune_epochs 8 \
    --batch_size 32 \
    --out_dir /content/output_models


2025-11-26 10:48:07.058543: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764154087.083236    2451 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764154087.090488    2451 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764154087.108829    2451 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764154087.108877    2451 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764154087.108883    2451 computation_placer.cc:177] computation placer alr

In [7]:
with open('/content/train_script.py', 'r') as f:
    print(f.read())

import os
import argparse
import tensorflow as tf
import json

# --- Paste your entire training script here ---
# (everything from build_dataset(), build_model(), fine_tune_model(), main(), etc.)



In [ ]:
%%writefile /content/train_script.py
import os
import json
import argparse
import tensorflow as tf
import tensorflow_datasets as tfds

def build_dataset(img_size, batch_size, model_type):
    # Load dataset info
    info = tfds.builder("plant_village").info
    class_names = info.features['label'].names
    num_classes = info.features['label'].num_classes

    # Load and shuffle dataset
    ds = tfds.load("plant_village", split="train", as_supervised=True, shuffle_files=True)

    # ---- SPEED BOOST: LIMIT DATASET ----
    # Full dataset = 54,303 images → Too slow on CPU.
    # Use first 8,000 images for fast but accurate training.
    ds = ds.take(8000)

    # Train/Val split
    train_ds = ds.take(6400)   # 80%
    val_ds   = ds.skip(6400)   # 20%

    # Light augmentation (fast)
    aug = tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal')
    ])
    preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

    def map_train_fn(image, label):
        image = tf.image.resize(image, img_size)
        image = aug(image)
        image = preprocess_input(image)
        label = tf.one_hot(label, num_classes) if model_type == "disease" else tf.cast(label, tf.float32)
        return image, label

    def map_val_fn(image, label):
        image = tf.image.resize(image, img_size)
        image = preprocess_input(image)
        label = tf.one_hot(label, num_classes) if model_type == "disease" else tf.cast(label, tf.float32)
        return image, label

    train_ds = train_ds.map(map_train_fn, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    val_ds   = val_ds.map(map_val_fn, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return train_ds, val_ds, num_classes, class_names

def build_model(num_classes, img_size, model_type):
    base = tf.keras.applications.MobileNetV2(
        include_top=False, input_shape=img_size + (3,), weights="imagenet"
    )
    base.trainable = False

    inputs = tf.keras.layers.Input(shape=img_size + (3,))
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    if model_type == "disease":
        outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
        loss = "categorical_crossentropy"
    else:
        outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
        loss = "binary_crossentropy"

    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=loss, metrics=["accuracy"])
    return model

def fine_tune_model(model, lr=1e-5):
    model.layers[1].trainable = True
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss=model.loss, metrics=["accuracy"])
    return model

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--model_type', required=True, choices=['gate', 'disease'])
    parser.add_argument('--img_size', type=int, default=224)
    parser.add_argument('--epochs', type=int, default=15)
    parser.add_argument('--fine_tune_epochs', type=int, default=8)
    parser.add_argument('--batch_size', type=int, default=32)
    parser.add_argument('--out_dir', default='/content/output_models')
    args = parser.parse_args()

    img_size = (args.img_size, args.img_size)
    os.makedirs(args.out_dir, exist_ok=True)

    # Build dataset
    train_ds, val_ds, num_classes, class_names = build_dataset(
        img_size, args.batch_size, args.model_type
    )

    # Build model
    model = build_model(num_classes, img_size, args.model_type)

    print("\n📌 Model Summary:")
    model.summary()

    # Train (initial)
    print("\n🚀 Starting training...")
    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=args.epochs,
        callbacks=[tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=3, restore_best_weights=True
        )],
    )

    # Fine-tune
    print("\n🔧 Fine-tuning model...")
    model = fine_tune_model(model)
    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=args.fine_tune_epochs,
        callbacks=[tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=3, restore_best_weights=True
        )],
    )

    # Save model
    model_path = os.path.join(args.out_dir, f"{args.model_type}_model.h5")
    model.save(model_path)
    print(f"\n✅ Model saved: {model_path}")

    # Save class names
    class_file = os.path.join(args.out_dir, f"{args.model_type}_class_names.json")
    with open(class_file, "w") as f:
        json.dump(class_names, f)
    print(f"📁 Class names saved: {class_file}")

    # Evaluate
    loss, acc = model.evaluate(val_ds)
    print(f"\n🎯 Final Validation — Loss: {loss:.4f}, Accuracy: {acc:.4f}")

if __name__ == "__main__":
    main()


Overwriting /content/train_script.py


In [8]:
import os
import shutil

raw_dir = "/content/raw_images"  # Where you put all your downloaded images
dataset_dir = "/content/dataset"

classes = ["Healthy", "Disease1", "Disease2", "Not_Plant"]

# Create folders if not exist
for c in classes:
    os.makedirs(os.path.join(dataset_dir, c), exist_ok=True)

# Move images based on filename keywords
for file in os.listdir(raw_dir):
    if not file.lower().endswith((".jpg", ".png", ".jpeg")):
        continue
    moved = False
    for c in classes:
        if c.lower() in file.lower():
            shutil.move(os.path.join(raw_dir, file), os.path.join(dataset_dir, c))
            moved = True
            break
    if not moved:
        shutil.move(os.path.join(raw_dir, file), os.path.join(dataset_dir, "Not_Plant"))

print("Dataset ready!")


FileNotFoundError: [Errno 2] No such file or directory: '/content/raw_images'

In [ ]:
!python /content/train_script.py \
  --model_type disease \
  --img_size 224 \
  --epochs 15 \
  --fine_tune_epochs 8 \
  --batch_size 32 \
  --out_dir /content/output_models

2025-11-25 12:12:24.301584: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764072744.328078   20851 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764072744.335313   20851 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764072744.353566   20851 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764072744.353624   20851 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764072744.353628   20851 computation_placer.cc:177] computation placer alr

In [ ]:
!ls /content/output_models


disease_class_names.json  disease_model.h5


In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image
import json

# Load model
model = tf.keras.models.load_model("/content/output_models/disease_model.h5")

# Load class names (list!)
with open("/content/output_models/disease_class_names.json", "r") as f:
    class_map = json.load(f)   # <-- This is a list

# Choose an image
img_path = "/content/images (1).jpg"   # Replace with your actual file

# Preprocess
img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Prediction
pred = model.predict(img_array)[0]
class_id = np.argmax(pred)
confidence = pred[class_id]

# Output using list index
predicted_class = class_map[class_id]

print("Prediction:", predicted_class)
print("Confidence:", confidence)


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Prediction: Cherry___Powdery_mildew
Confidence: 0.59401506
